# Practical 4 – Anomaly Detection using Autoencoder

**Aim:** Train an Autoencoder using digit-zero images, reconstruct test images, and flag samples with high reconstruction error.

Every important code line is explained using `# comments` directly inside the code. Run the cells from top to bottom.

## 1. Import libraries and load data

In [ ]:
# TensorFlow is used to build and train the Autoencoder.
import tensorflow as tf

# NumPy calculates reconstruction errors.
import numpy as np

# Matplotlib plots reconstruction errors and the threshold.
import matplotlib.pyplot as plt

# Model connects the input and output; Input and Dense define the network layers.
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# Load the MNIST handwritten digit dataset.
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Print the original dataset dimensions.
print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)

## 2. Keep only digit zero and normalize

In [ ]:
# Keep only training images whose label is 0; the model learns this normal pattern.
x_train = x_train[y_train == 0]

# Scale pixel values from 0–255 into the 0–1 range.
x_train = x_train / 255.0
x_test = x_test / 255.0

# Display the number and shape of digit-zero training images.
print("Digit 0 training images:", x_train.shape)

## 3. Flatten images

In [ ]:
# Convert each 28x28 image into a vector of 784 pixel values.
# -1 tells NumPy to calculate the number of images automatically.
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# Display the new flattened dimensions.
print("Flattened training data:", x_train.shape)
print("Flattened testing data:", x_test.shape)

## 4. Build encoder and decoder

In [ ]:
# Define an input that receives 784 pixel values per image.
input_layer = Input(shape=(784,))

# ENCODER: progressively compress the image representation.
encoded = Dense(16, activation='relu')(input_layer)  # Compress 784 values into 16 learned features.
encoded = Dense(8, activation='relu')(encoded)        # Compress 16 features into 8.
latent = Dense(4, activation='relu')(encoded)         # Bottleneck/latent representation contains 4 values.

# DECODER: expand the compressed representation back to the original size.
decoded = Dense(8, activation='relu')(latent)          # Expand 4 values into 8.
decoded = Dense(16, activation='relu')(decoded)       # Expand 8 values into 16.
output_layer = Dense(784, activation='sigmoid')(decoded)  # Reconstruct 784 pixel values between 0 and 1.

# Connect the input to the reconstructed output to create the Autoencoder.
autoencoder = Model(input_layer, output_layer)

# Display the model architecture.
autoencoder.summary()

## 5. Compile and train

In [ ]:
# Configure the optimizer, reconstruction loss, and additional error metric.
autoencoder.compile(
    optimizer='adam',  # Adam updates weights to improve reconstruction.
    loss='mse',         # Mean Squared Error measures squared pixel reconstruction differences.
    metrics=['mae']     # Mean Absolute Error reports average absolute pixel difference.
)

In [ ]:
# Train the Autoencoder to reproduce each input image.
history = autoencoder.fit(
    x_train,        # Input images containing only digit zero.
    x_train,        # Target is the same image: learn to reconstruct the input.
    epochs=10,      # Train for 10 complete passes.
    batch_size=32   # Process 32 images before each weight update.
)

## 6. Reconstruct and calculate error

In [ ]:
# Pass test images through the trained model to obtain reconstructed images.
prediction = autoencoder.predict(x_test)

# Calculate one mean squared reconstruction error for each test image.
error = np.mean((x_test - prediction) ** 2, axis=1)

# Display how many test images have reconstruction-error values.
print('Number of reconstruction errors:', len(error))

## 7. Set threshold and identify anomalies

In [ ]:
# Set the threshold at the 95th percentile of the calculated test errors.
threshold = np.percentile(error, 95)

# Mark an image as an anomaly when its error is greater than the threshold.
anomalies = error > threshold

# Print the threshold and counts of flagged and unflagged images.
print("Threshold:", threshold)
print("Anomaly:", sum(anomalies))
print("Normal Records:", sum(~anomalies))  # ~ reverses True/False to count the remaining images.

## 8. Plot anomaly scores

In [ ]:
# Plot each test image's reconstruction error.
plt.plot(error)

# Draw a horizontal line showing the anomaly threshold.
plt.axhline(threshold, color='red')

# Add graph title and axis labels, then display the graph.
plt.title('Anomaly Detection')
plt.xlabel('Images')
plt.ylabel('Reconstruction Error')
plt.show()